In [1]:
import sys

print("Python version:")
print(sys.version)

import pandas as pd
import numpy as np
import sklearn

print("\nLibraries imported successfully!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python version:
3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]

Libraries imported successfully!
Pandas: 2.2.2
NumPy: 1.26.4
Scikit-learn: 1.5.1


In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../data/Blackout_PowerGraph_1000.csv")

# Display basic information
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

Dataset shape: (1000, 225)

First 5 rows:


,bus_1_net_active_power,bus_2_net_active_power,bus_3_net_active_power,bus_4_net_active_power,bus_5_net_active_power,bus_6_net_active_power,bus_7_net_active_power,bus_8_net_active_power,bus_9_net_active_power,bus_10_net_active_power,...,line_30_rating,line_31_rating,line_32_rating,line_33_rating,line_34_rating,line_35_rating,line_36_rating,line_37_rating,line_38_rating,blackout
0,-0.033243,-0.019466,-0.228603,-0.095843,-0.092085,-0.173495,-0.033296,-0.217331,-0.222341,-0.247390,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0
1,-0.026204,-0.008625,-0.195121,-0.081933,-0.078730,-0.148137,-0.042461,-0.185511,-0.189782,-0.211138,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0
2,-0.034218,-0.020822,-0.222321,-0.093231,-0.089577,-0.168736,-0.034644,-0.211360,-0.216231,-0.240588,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0
3,-0.036552,-0.024058,-0.207475,-0.087072,-0.083664,-0.157496,-0.037863,-0.197252,-0.201796,-0.224513,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,1
4,0.029068,0.042205,-0.216479,-0.089889,-0.086306,-0.163932,0.040985,-0.205731,-0.210508,-0.234393,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 225 entries, bus_1_net_active_power to blackout
dtypes: float64(224), int64(1)
memory usage: 1.7 MB


In [3]:
# Count each class
class_counts = df["blackout"].value_counts().sort_index()

# Calculate percentages
class_percentages = df["blackout"].value_counts(normalize=True).sort_index() * 100

print("Class distribution:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages.round(2))

print("\nMissing values:", df.isnull().sum().sum())

Class distribution:
blackout
0    798
1    202
Name: count, dtype: int64

Class percentages:
blackout
0    79.8
1    20.2
Name: proportion, dtype: float64

Missing values: 0


In [4]:
# Separate input features and target variable
X = df.drop(columns=["blackout"])
y = df["blackout"]

# Check duplicated input feature rows
duplicate_count = X.duplicated().sum()

print("Duplicated feature rows:", duplicate_count)

# Check whether duplicated features have consistent labels
duplicate_data = df[df.duplicated(
    subset=X.columns,
    keep=False
)]

if len(duplicate_data) > 0:
    label_counts = duplicate_data.groupby(
        list(X.columns)
    )["blackout"].nunique()

    print(
        "Duplicated feature groups with conflicting labels:",
        (label_counts > 1).sum()
    )
else:
    print("No duplicated feature rows found.")

Duplicated feature rows: 0
No duplicated feature rows found.


In [5]:
from sklearn.model_selection import train_test_split

# First, separate the test set (15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Then, split the remaining data into training (70%) and validation (15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=y_train_val
)

# Display dataset sizes
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("\nTotal samples:", len(X_train) + len(X_val) + len(X_test))

Training set: (700, 224)
Validation set: (150, 224)
Test set: (150, 224)

Total samples: 1000


In [6]:
# Check class distribution in each dataset
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name} set:")
    print(labels.value_counts().sort_index())
    print((labels.value_counts(normalize=True).sort_index() * 100).round(2))


Training set:
blackout
0    558
1    142
Name: count, dtype: int64
blackout
0    79.71
1    20.29
Name: proportion, dtype: float64

Validation set:
blackout
0    120
1     30
Name: count, dtype: int64
blackout
0    80.0
1    20.0
Name: proportion, dtype: float64

Test set:
blackout
0    120
1     30
Name: count, dtype: int64
blackout
0    80.0
1    20.0
Name: proportion, dtype: float64


In [7]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the baseline model
baseline_model = DecisionTreeClassifier(random_state=42)

# Train the model
baseline_model.fit(X_train, y_train)

print("Decision Tree baseline trained successfully!")

Decision Tree baseline trained successfully!


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Make predictions on the validation set
y_val_pred = baseline_model.predict(X_val)
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]

# Calculate evaluation metrics
print("Validation Set Evaluation")
print("--------------------------")

print("Accuracy:", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, y_val_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_val, y_val_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_val, y_val_proba), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, zero_division=0))

Validation Set Evaluation
--------------------------
Accuracy: 0.9
Precision: 0.7586
Recall: 0.7333
F1-score: 0.7458
ROC-AUC: 0.8375

Confusion Matrix:
[[113   7]
 [  8  22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.94       120
           1       0.76      0.73      0.75        30

    accuracy                           0.90       150
   macro avg       0.85      0.84      0.84       150
weighted avg       0.90      0.90      0.90       150

